# Evaluate and Use Models from Model Garden

## Task 2. Prepare prompts for model evaluation

In [1]:
!pip install --upgrade --quiet google-cloud-aiplatform \
    google-cloud-logging litellm 'anthropic[vertex]'

### Setup and Initialize Vertex AI 

Run the following cells to install and import packages and initialize Vertex AI.

In [2]:
import google.cloud.logging
from google.cloud import aiplatform
from IPython.display import Markdown
import logging
import pandas as pd
import random

import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from vertexai.evaluation import (
    MetricPromptTemplateExamples,
    EvalTask,
    PairwiseMetric,
    PointwiseMetric,
)

# Sets Pandas to display content in wider columns
pd.set_option("display.max_colwidth", None)

# Do not remove logging section
client = google.cloud.logging.Client()
client.setup_logging()

# TODO Add your project and location to initialize vertxai
PROJECT_ID, = !gcloud config get project
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

pd.set_option("display.max_colwidth", None)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.appengine_logging_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.appengine_logging_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_pytho

### Define Prompts for Evaluation

In [4]:
prompt_template = """
    You are a technical architect.

    Prepare a high-level project plan for how a solution to the
    following PROJECT REQUEST could be developed using Google Cloud.
    
    We don't want to begin projects that we can't deliver well in
    under four months, so also weigh in on whether you think this
    is a good time for us to attempt this project.
    
    PROJECT REQUEST:
    {project_request}
"""

In [5]:
project_requests = [
    """
    Inventory Management with Real-time Visibility.
    Retail division needs to track inventory across all physical stores
    and e-commerce channels with real-time accuracy. Key pain point
    is stockouts and overstock leading to lost sales and increased
    holding costs. They're looking for a cloud solution that integrates
    seamlessly with existing POS systems and offers predictive analytics
    for demand forecasting.
    """,
    """
    AI-Driven Material Selection & Performance Prediction.
    Product Development is interested in AI's ability to analyze material properties
    and predict their performance under various conditions within the
    context of their industrial designs. Wants assistance in identifying
    the most suitable and cost-effective materials early in the design process.
    """,
    """
    Optimize Yield & Reduce Input Costs.
    Agriculture department specifically mentioned a desire to increase crop
    output (e.g., by X%) while simultaneously decreasing expenses on water,
    fertilizer, and pesticides (e.g., by Y%) through more precise application
    and predictive insights.
    """,
    """
    Enhanced Predictive Maintenance for Fleet Operations.
    Distribution department expressed a strong need to leverage AI/ML on their
    cloud platform for proactive identification of potential component failures
    across their delivery fleet. Key drivers are reducing unscheduled downtime,
    optimizing maintenance schedules, and minimizing operational disruption.
    Requires ingestion of vast amounts of sensor data (IoT), flight logs, and
    maintenance records.
    """,
    """
    Inaccurate Vehicle Valuation.
    Distribution department struggles with inconsistent and often delayed real-time
    pricing for used vehicles, leading to lost profit margins and slower inventory
    turnover. Needs a solution that leverages AI to provide highly accurate,
    dynamic valuation based on a wide range of data points (market trends,
    condition, history, local demand).
    """

]

In [6]:
# TODO
# Update the code so that the variable `prompts` contains a list of
# strings, applying the `prompt_template` to each item in `project_requests`

prompts = [prompt_template.format(project_request=request) for request in project_requests]

print(prompts[0])


    You are a technical architect.

    Prepare a high-level project plan for how a solution to the
    following PROJECT REQUEST could be developed using Google Cloud.
    
    We don't want to begin projects that we can't deliver well in
    under four months, so also weigh in on whether you think this
    is a good time for us to attempt this project.
    
    PROJECT REQUEST:
    
    Inventory Management with Real-time Visibility.
    Retail division needs to track inventory across all physical stores
    and e-commerce channels with real-time accuracy. Key pain point
    is stockouts and overstock leading to lost sales and increased
    holding costs. They're looking for a cloud solution that integrates
    seamlessly with existing POS systems and offers predictive analytics
    for demand forecasting.
    



In [7]:
# Check your work

assert len(prompts) == 5, "You don't have a response for each project_request."
assert type(prompts[0]) == str, ("Make sure your prompts are strings.")

## Task 3. Get Qwen3-Next Instruct Responses using the LiteLLM SDK

In [8]:
import os
import litellm

MAX_TOKENS = 1024

In [9]:
# Vertex AI uses application default credentials
# If not using a Vertex AI resource, use 'gcloud auth application-default login' to authenticate
# Here we unpack from the existing env variable GOOGLE_CLOUD_PROJECT
os.environ["VERTEXAI_PROJECT"] = os.environ["GOOGLE_CLOUD_PROJECT"]
os.environ["VERTEXAI_LOCATION"] = "global"

In [10]:
# Non-streaming response
response = litellm.responses(
    model="vertex_ai/qwen/qwen3-next-80b-a3b-instruct-maas",
    input="Tell me a three sentence bedtime story about a unicorn.",
    max_output_tokens=MAX_TOKENS
)

print(response.output[0].content[0].text)

17:55:52 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= qwen/qwen3-next-80b-a3b-instruct-maas; provider = vertex_ai
17:55:54 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
17:55:54 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler


Under a silver moon, a tiny unicorn named Lumi glowed softly as she wandered the sleepy meadow, collecting lost starlight in her horn to brighten the dreams of sleeping children. She brushed her velvet muzzle against each window, whispering lullabies made of wind chimes and dandelion fluff, until the whole world sighed into sweet slumber. As dawn tiptoed near, Lumi curled beneath a clover tree, her glow fading gently—knowing tomorrow, she’d dream of them too.


In [11]:
qwen3_responses = []

# TODO
# Update the code below to populate the list qwen3_responses
# with just the text string of Qwen3-Next Instruct's response to each prompt

for prompt in prompts:
    q3_response = litellm.responses(
        model="vertex_ai/qwen/qwen3-next-80b-a3b-instruct-maas",
        input=prompt,
        max_output_tokens=MAX_TOKENS
    )
    qwen3_responses.append(q3_response.output[0].content[0].text)

17:55:54 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= qwen/qwen3-next-80b-a3b-instruct-maas; provider = vertex_ai
17:55:59 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
17:55:59 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
17:55:59 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= qwen/qwen3-next-80b-a3b-instruct-maas; provider = vertex_ai
17:56:04 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
17:56:04 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
17:56:04 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= qwen/qwen3-next-80b-a3b-instruct-maas; provider = vertex_ai
17:56:08 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
17:56:08 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
17:56:09 - LiteLLM:INFO: utils.py:3895 - 
LiteLLM completion() model= 

In [12]:
# Check your work

assert len(prompts) == len(qwen3_responses), "You don't have a response for each prompt."
assert type(qwen3_responses[0]) == str, ("Extract just the text response, "
                                          "not the entire response object.")
# View one response
Markdown(qwen3_responses[0])

Absolutely — as a Technical Architect, I’ll provide a **high-level project plan** for delivering **Inventory Management with Real-time Visibility** on **Google Cloud**, along with a **go/no-go assessment** based on your 4-month delivery constraint.

---

## ✅ **PROJECT ASSESSMENT: GO / NO-GO**

### **VERDICT: ✅ GO — WITH CONDITIONS**

> **This is a viable 4-month project** — *but only if you treat it as a phased MVP launch*, not a full enterprise overhaul.  
> The core requirements (real-time inventory + POS integration + predictive analytics) are well-supported by Google Cloud and modern architectures.  
> **Critical success factor**: You must **limit scope to 3–5 pilot stores + e-commerce channel** in Month 1–2, then scale.

**Why it’s feasible in 4 months:**
- Google Cloud offers fully managed services for ingestion, storage, analytics, and ML.
- POS integrations are typically API-based (REST/GraphQL), not legacy batch systems.
- Predictive demand forecasting can start with a simple, data-driven model (not AI-overkill).
- No need to replace existing POS — integrate *alongside*.

**Risks to Mitigate:**
- Poor data quality from legacy POS systems.
- Resistance from store ops teams to real-time data adoption.
- Over-engineering the forecasting model (start simple!).

---

## 🚀 HIGH-LEVEL PROJECT PLAN (4 MONTHS)

### **PHASE 0: PREP (Week 1–2) — Discovery & Scope Lock**
**Goal:** Define MVP scope, identify data sources, secure stakeholder buy-in.

- **Stakeholder Workshops**: Align with Retail Ops, IT, E-commerce, and Finance teams.
- **MVP Scope Definition**:
  - **Stores**: 3–5 high-volume pilot locations (not all 50+).
  - **Channels**: E-commerce + physical POS (online + in-store sales).
  - **Data Sources**: POS systems (e.g., Square, Shopify, Oracle Retail), warehouse systems (if any), historical sales data.
  - **Out of Scope**: Supplier integration, automatic reordering, mobile app for staff (Phase 2).
- **Data Audit**: Validate POS API access, data frequency (real-time? hourly?), data quality (missing SKUs? inconsistent IDs?).
- **Tool Selection**:
  - **Data Ingestion**: Pub/Sub + Cloud Dataflow
  - **Storage**: BigQuery (for analytics) + Cloud Storage (for raw logs)
  - **Real-time Dashboard**: Looker Studio (embedded in internal portal)
  - **ML Forecasting**: BigQuery ML (simple time-series) or Vertex AI (if complex modeling needed)
  - **Integration**: Cloud Run (for lightweight APIs), Cloud Functions (for event triggers)
  - **Security & IAM**: Cloud IAM + VPC Service Controls

> 💡 *Use BigQuery ML instead of Vertex AI initially — faster to prototype, no ML Ops overhead.*

---

### **PHASE 1: BUILD (Week 3–8) — Core Pipeline + MVP Dashboard**
**Goal**: Build end-to-end real-time inventory pipeline and basic dashboard.

#### **Architecture Flow**:
```
[POS Systems] → (API/Webhook) → [Cloud Pub/Sub] → [Cloud Dataflow (Stream)] → [BigQuery (Real-time Table)]
                                                       ↓
[Historical Sales Data] → [BigQuery (Batch Table)] → [BigQuery ML Model] → [Forecast Table]
                                                       ↓
[Looker Studio Dashboard] ← [BigQuery] ← [Forecast + Real-time Stock]
```

#### **Key Deliverables**:
- ✅ **Real-time inventory table** in BigQuery updated every 5–15 mins (via Dataflow).
- ✅ **Looker Studio dashboard** showing:
  - Current stock levels per SKU per store
  - Stockout alerts (if stock < min threshold)
  - Overstock highlights (stock > 90-day avg)
- ✅ **Basic forecast model**: Prophet or ARIMA via BigQuery ML — predicts next 7-day demand per SKU per store.
- ✅ **Alerting**: Cloud Functions trigger email/SMS (via Gmail API or Twilio) when stock < threshold.

> ⚡ **Pro Tip**: Use `BQ ML` with `ARIMA_PLUS` — it auto-detects seasonality and trends. No data science team needed.

---

### **PHASE 2: INTEGRATE & VALIDATE (Week 9–12) — Pilot Launch & Feedback**
**Goal**: Deploy to pilot stores, validate accuracy, refine alerts.

- Deploy to 3–5 pilot stores.
- Train store managers on dashboard + alert response.
- Collect feedback: Are alerts useful? Are stock levels accurate? Any data gaps?
- **Measure KPIs**:
  - Reduction in

## Task 4. Get Claude Responses using the Anthropic SDK

In [13]:
from anthropic import AnthropicVertex

In [14]:
CLAUDE_MODEL = "claude-sonnet-4@20250514"
CLAUDE_LOCATIONS = ["us-east5", "europe-west4", "GLOBAL"]

# TODO
# Implement an anthropic_client to access the Claude Sonnet 4 model
# using the model ID defined above

anthropic_client = AnthropicVertex(project_id=PROJECT_ID, region=CLAUDE_LOCATIONS[0])

In [16]:
claude_responses = []

# TODO
# Update the code below to populate the list claude_responses
# with just the text string of Clade Sonnet 4's response to each prompt

for prompt in prompts:
    message = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": prompt}],
    )
    claude_response = message.content[0].text
    claude_responses.append(claude_response)

In [17]:
# Check your work

assert len(prompts) == len(claude_responses), "You don't have a response for each prompt."
assert type(claude_responses[0]) == str, ("Extract just the text response, "
                                          "not the entire response object.")

# View one response
Markdown(claude_responses[0])

# High-Level Project Plan: Real-time Inventory Management Solution

## Executive Summary
This project is **feasible within 4 months** with proper scope management and phased delivery. The solution leverages Google Cloud's real-time data processing capabilities and ML services to create a unified inventory visibility platform.

## Solution Architecture Overview

### Core Google Cloud Components
- **Cloud SQL/Firestore**: Central inventory database
- **Pub/Sub**: Real-time event streaming from POS systems
- **Dataflow**: Stream processing for inventory updates
- **BigQuery**: Data warehouse for analytics and reporting
- **Vertex AI**: Demand forecasting and predictive analytics
- **Cloud Run**: Microservices for API layer
- **Cloud Load Balancing**: High availability
- **Cloud Monitoring**: System observability

## 4-Month Phased Delivery Plan

### Phase 1: Foundation (Month 1)
**Weeks 1-2: Infrastructure & Data Architecture**
- Set up GCP project structure and IAM
- Design data models for inventory, transactions, locations
- Establish CI/CD pipelines (Cloud Build)
- Create development environments

**Weeks 3-4: Core Data Ingestion**
- Implement Pub/Sub topics for POS integration
- Build Dataflow pipelines for real-time inventory updates
- Set up Cloud SQL database with initial schema
- Create basic API endpoints (Cloud Run)

### Phase 2: Integration & Real-time Processing (Month 2)
**Weeks 5-6: POS Integration**
- Develop connectors for existing POS systems
- Implement real-time inventory adjustment workflows
- Build data validation and error handling
- Set up monitoring and alerting

**Weeks 7-8: Multi-channel Synchronization**
- E-commerce platform integration
- Cross-channel inventory reconciliation
- Implement business rules engine
- Basic dashboard development

### Phase 3: Analytics & Intelligence (Month 3)
**Weeks 9-10: Data Warehouse & Reporting**
- BigQuery data mart implementation
- Historical data migration and ETL
- Standard reporting dashboard creation
- Performance optimization

**Weeks 11-12: Predictive Analytics MVP**
- Vertex AI model development for demand forecasting
- Basic stockout/overstock predictions
- Integration with inventory replenishment workflows
- Initial ML model training and validation

### Phase 4: Enhancement & Production (Month 4)
**Weeks 13-14: Advanced Features**
- Refined predictive models
- Advanced alerting and notifications
- Mobile-responsive interfaces
- Performance tuning

**Weeks 15-16: Go-Live Preparation**
- User acceptance testing
- Production deployment
- Staff training and documentation
- Monitoring and support procedures

## Technical Implementation Strategy

### Real-time Data Flow
```
POS Systems → Pub/Sub → Dataflow → Cloud SQL/Firestore
                    ↓
              BigQuery ← Batch ETL
                    ↓
              Vertex AI → Predictions → Dashboard
```

### Integration Approach
- **API-first design** using OpenAPI specifications
- **Event-driven architecture** for real-time updates
- **Microservices pattern** for scalability
- **Standard retail protocols** (ARTS, RIC) where applicable

### Predictive Analytics Components
- **Demand forecasting**: Time series analysis using AutoML
- **Seasonality detection**: Historical sales pattern analysis
- **Stockout prediction**: Inventory velocity + lead time modeling
- **Reorder optimization**: Economic order quantity with ML enhancement

## Risk Assessment & Mitigation

### High Risks
1. **POS System Integration Complexity**
   - *Mitigation*: Early discovery phase, API documentation review
   - *Fallback*: Batch integration for legacy systems

2. **Data Quality Issues**
   - *Mitigation*: Implement robust validation and cleansing
   - *Plan*: Gradual rollout with data quality monitoring

3. **Change Management**
   - *Mitigation*: Early stakeholder engagement, training programs

### Medium Risks
- Performance at scale (addressed with GCP auto-scaling)
- ML model accuracy (iterative improvement approach)

## Resource Requirements

### Team Structure (Recommended)
- **Project Manager**: 1 FTE
- **Cloud Architect**: 1 FTE
- **Backend Developers**: 2 FTE
- **Data Engineer**: 1 FTE
- **ML Engineer**:

## Task 5. Conduct a pairwise evaluation

### Explore Metrics and Select Evaluation Criteria

In [18]:
MetricPromptTemplateExamples.list_example_metric_names()

['coherence',
 'fluency',
 'safety',
 'groundedness',
 'instruction_following',
 'verbosity',
 'text_quality',
 'summarization_quality',
 'question_answering_quality',
 'multi_turn_chat_quality',
 'multi_turn_safety',
 'pairwise_coherence',
 'pairwise_fluency',
 'pairwise_safety',
 'pairwise_groundedness',
 'pairwise_instruction_following',
 'pairwise_verbosity',
 'pairwise_text_quality',
 'pairwise_summarization_quality',
 'pairwise_question_answering_quality',
 'pairwise_multi_turn_chat_quality',
 'pairwise_multi_turn_safety']

In [19]:
print(MetricPromptTemplateExamples.Pairwise.TEXT_QUALITY.metric_prompt_template)


# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by two AI models. We will provide you with the user input and a pair of AI-generated responses (Response A and Response B). You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will first judge responses individually, following the Rating Rubric and Evaluation Steps. Then you will give step-by-step explanations for your judgment, compare the results to declare the winner based on the Rating Rubric and Evaluation Steps.

# Evaluation
## Metric Definition
You will be assessing the Text Quality of each model's response, which measures how effectively the text conveys clear, accurate, and engaging information that directly addresses the user's prompt, considering factors like fluency, coherence, relevance, and conciseness.

## Criteria
Coherence: The re

### Prepare and Execute the Evaluation Task

In [33]:
# TODO
# Provide all the variables required for the metric
# selected above

eval_dataset=pd.DataFrame(
    {
      "prompt": prompts,
      "baseline_model_response": qwen3_responses,
      "response": claude_responses,
    }
)
eval_dataset.head(1)

prompt  \
0  \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    Inventory Management with Real-time Visibility.\n    Retail division needs to track inventory across all physical stores\n    and e-commerce channels with real-time accuracy. Key pain point\n    is stockouts and overstock leading to lost sales and increased\n    holding costs. They're looking for a cloud solution that integrates\n    seamlessly with existing POS systems and offers predictive analytics\n    for demand forecasting.\n    \n   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [32]:
selected_metric = MetricPromptTemplateExamples.Pairwise.TEXT_QUALITY

# TODO
# Complete the definition of the EvalTask using the dataset defined
# above and the selected metric. Keep the provided experiment name.

pairwise_eval_task = EvalTask(
    dataset=eval_dataset,
    metrics=[selected_metric],
    experiment="pairwise-text-quality-001",
)

In [22]:
pairwise_result = pairwise_eval_task.evaluate()

Associating projects/80995366258/locations/us-central1/metadataStores/default/contexts/pairwise-text-quality-001-7afa93b7-194f-47e7-982d-21542aaa5694 to Experiment: pairwise-text-quality-001


Computing metrics with a total of 5 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 5/5 [00:23<00:00,  4.77s/it]

All 5 metric requests are successfully computed.
Evaluation Took:23.9501343070001 seconds


In [23]:
# TODO
# Display the result's summary_metrics
pairwise_result.summary_metrics

{'row_count': 5,
 'pairwise_text_quality/candidate_model_win_rate': np.float64(0.4),
 'pairwise_text_quality/baseline_model_win_rate': np.float64(0.6)}

In [24]:
# TODO
# Display the result's metrics_table table
pairwise_result.metrics_table

prompt  \
0                                               \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    Inventory Management with Real-time Visibility.\n    Retail division needs to track inventory across all physical stores\n    and e-commerce channels with real-time accuracy. Key pain point\n    is stockouts and overstock leading to lost sales and increased\n    holding costs. They're looking for a cloud solution that integrates\n    seamlessly with existing POS systems and offers predictive analytics\n    for demand forecasting.\n    \n   
1                                                                                                                  \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    AI-Driven Material Selection & Performance Prediction.\n    Product Development is interested in AI's ability to analyze material properties\n    and predict their performance under various conditions within the\n    context of their industrial designs. Wants assistance in identifying\n    the most suitable and cost-effective materials early in the design process.\n    \n   
2                                                                                                                                                                                     \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    Optimize Yield & Reduce Input Costs.\n    Agriculture department specifically mentioned a desire to increase crop\n    output (e.g., by X%) while simultaneously decreasing expenses on water,\n    fertilizer, and pesticides (e.g., by Y%) through more precise application\n    and predictive insights.\n    \n   
3  \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    Enhanced Predictive Maintenance for Fleet Operations.\n    Distribution department expressed a strong need to leverage AI/ML on their\n    cloud platform for proactive identification of potential component failures\n    across their delivery fleet. Key drivers are reducing unscheduled downtime,\n    optimizing maintenance schedules, and minimizing operational disruption.\n    Requires ingestion of vast amounts of sensor data (IoT), flight logs, and\n    maintenance records.\n    \n   
4                                                                                        \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think 

In [25]:
# Explore the first result's explanation
print(pairwise_result.metrics_table.loc[0,'pairwise_text_quality/explanation'])

BASELINE response provides a more insightful and practical project plan, especially regarding the 4-month time constraint. Its 'GO - WITH CONDITIONS' assessment is superior, offering specific scope limitations (MVP, pilot stores) and expert advice (e.g., using BigQuery ML for faster prototyping). The architecture flow is also clearer and more directly addresses real-time inventory. Both responses cut off prematurely, but BASELINE response's content before the cutoff is of higher quality and more directly addresses the user's specific constraints and requests.


## Test the CodeGemma deployment

In [ ]:
import google.cloud.logging
import logging

from google.cloud import aiplatform

# Do not remove logging section
client = google.cloud.logging.Client()
client.setup_logging()

# Replace your project number and endpoint id
ENDPOINT_ID = "mg-endpoint-e0c603e0-68fe-471e-a7ae-315c7ceec3b9"

endpoint_resource_name=f"projects/{PROJECT_ID}/locations/us-central1/endpoints/{ENDPOINT_ID}"
endpoint=aiplatform.Endpoint(endpoint_resource_name)

In [43]:

prompt = "Write a function to list n Fibonacci numbers in Python." 
max_tokens = 500 
temperature = 1.0 
top_p = 1.0  
top_k = 1 
raw_response = True

instances = [
    {
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "raw_response": raw_response,
    },
]

response = endpoint.predict(
    instances=instances
)

# "<|file_separator|>" is the end of the file token.
for prediction in response.predictions:
    print(prediction.split("<|file_separator|>")[0])


print(response.deployed_model_id)

# Do not remove logging section
log_message = f"Fibonacci function: {response}"
logging.info(log_message)

py
<|fim_prefix|><|fim_suffix|><|fim_middle|>def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    else:
        fib_sequence = [0, 1]
        while len(fib_sequence) < n:
            next_number = fib_sequence[-1] + fib_sequence[-2]
            fib_sequence.append(next_number)
        return fib_sequence

n = int(input("Enter the number of Fibonacci numbers to generate: "))
fibonacci_numbers = fibonacci(n)
print("Fibonacci sequence:", fibonacci_numbers)

4599917395719487488


## Perform Cleanup

In [48]:
# Delete all endpoints and their deployed models
endpoints = aiplatform.Endpoint.list()

if not endpoints:
    print("No endpoints found in this project/region.")
else:
    for endpoint in endpoints:
        print(f"\nEndpoint: {endpoint.display_name} ({endpoint.resource_name})")

        deployed_models = endpoint.list_models()
        for dm in deployed_models:
            print(f"  Undeploying model ID: {dm.id}")            
            # TODO Uncomment the statement below to undeploy a model
            endpoint.undeploy(deployed_model_id=dm.id, sync=True)
            print(f"  Undeployed model ID: {dm.id}")

        print("  Deleting endpoint...")
        endpoint.delete(sync=True)
        print(f"  Deleted endpoint: {endpoint.display_name}")

print("\nDeleting models...")
models = aiplatform.Model.list()

for model in models:
    print(f"  Deleting model: {model.display_name}")
    # TODO Uncomment the statement below to delete the model.
    model.delete(sync=True)
    print("  Deleted.")
    
# Do not remove logging section
log_message = "Deleted endpoints"
logging.info(log_message)



Endpoint: google_codegemma-2b-mg-one-click-deploy (projects/80995366258/locations/us-central1/endpoints/mg-endpoint-e0c603e0-68fe-471e-a7ae-315c7ceec3b9)
  Undeploying model ID: 4599917395719487488
Undeploying Endpoint model: projects/80995366258/locations/us-central1/endpoints/mg-endpoint-e0c603e0-68fe-471e-a7ae-315c7ceec3b9
Undeploy Endpoint model backing LRO: projects/80995366258/locations/us-central1/endpoints/mg-endpoint-e0c603e0-68fe-471e-a7ae-315c7ceec3b9/operations/5792273168324362240
Endpoint model undeployed. Resource name: projects/80995366258/locations/us-central1/endpoints/mg-endpoint-e0c603e0-68fe-471e-a7ae-315c7ceec3b9
  Undeployed model ID: 4599917395719487488
  Deleting endpoint...
Deleting Endpoint : projects/80995366258/locations/us-central1/endpoints/mg-endpoint-e0c603e0-68fe-471e-a7ae-315c7ceec3b9
Endpoint deleted. . Resource name: projects/80995366258/locations/us-central1/endpoints/mg-endpoint-e0c603e0-68fe-471e-a7ae-315c7ceec3b9
Deleting Endpoint resource: proj